# `core/base.py` — Design Walkthrough

This notebook walks through the six classes in `core/base.py` interactively. The goal is to get comfortable with the API and to verify by eye that the core machinery works correctly before building backends on top of it.

At the end, a 25-line statevector backend is built inline to show the State/Simulator split working end-to-end.

In [ ]:
import numpy as np
from qaravan.core.base import (
    Gate, Circuit, State, Simulator, Observable, NoiseModel,
    IncompatibleNoiseError, IncompatibleStateError, _embed_gate,
)

# Common matrices used throughout
H_MAT  = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
X_MAT  = np.array([[0, 1], [1,  0]], dtype=complex)
Z_MAT  = np.array([[1, 0], [0, -1]], dtype=complex)
CNOT_MAT = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], dtype=complex)

---
## 1. Gate

`Gate(name, indices, matrix, time=None)` — the atomic unit. Backend-agnostic: it carries a matrix but doesn't know how any simulator will use it.

In [ ]:
h  = Gate("H",  0,      H_MAT)
x  = Gate("X",  1,      X_MAT)
cx = Gate("CX", [0, 1], CNOT_MAT)

for g in [h, x, cx]:
    print(f"{g!s:16s}  num_sites={g.num_sites}  local_dim={g.local_dim}  time={g.time}")

In [ ]:
# Single int index is normalised to list[int]
print("h.indices :", h.indices)   # [0], not 0
print("cx.indices:", cx.indices)  # [0, 1]

In [ ]:
# dagger: conjugate-transpose + "†" suffix
# H is self-inverse, so H† should equal H
hd = h.dagger()
print("H  matrix:")
print(np.round(h.matrix.real, 4))
print("\nH† matrix (should be identical — H is self-inverse):")
print(np.round(hd.matrix.real, 4))
print("\nH†H = I?", np.allclose(hd.matrix @ h.matrix, np.eye(2)))

In [ ]:
# CNOT is also self-inverse (it's a Hermitian unitary)
cxd = cx.dagger()
print("CNOT† == CNOT?", np.allclose(cxd.matrix, cx.matrix))

# Something non-self-inverse: S gate
S_MAT = np.array([[1, 0], [0, 1j]], dtype=complex)
s  = Gate("S",  0, S_MAT)
sd = s.dagger()
print("\nS  matrix:\n", s.matrix)
print("S† matrix:\n", sd.matrix)
print("S†S = I?", np.allclose(sd.matrix @ s.matrix, np.eye(2)))

---
## 2. Circuit

`Circuit` is an ordered sequence of `Gate` objects. It supports composition, repetition, and `.dagger()`. The key structure it builds lazily is **layers** — sets of non-overlapping gates that can run in parallel.

In [ ]:
ghz = Circuit([h, cx])

print(ghz)
print(f"n={ghz.n}, len={len(ghz)}, layers={ghz.layers}  # None until construct_layers()")

In [ ]:
# construct_layers uses greedy packing: each gate goes into the earliest layer
# it fits without qubit conflicts.
# H(0) and CX(0,1) both touch qubit 0, so they must be in separate layers.
ghz.construct_layers()
for i, layer in enumerate(ghz.layers):
    print(f"Layer {i}: {[str(g) for g in layer]}")

In [ ]:
# A 4-qubit brickwall to see parallel packing in action.
# All 4 single-qubit H gates share no qubits → one layer.
# CX(0,1) and CX(2,3) are on disjoint qubit sets → same layer.
# CX(1,2) conflicts with both → its own layer.
bw = Circuit([
    Gate("H", 0, H_MAT), Gate("H", 1, H_MAT),
    Gate("H", 2, H_MAT), Gate("H", 3, H_MAT),
    Gate("CX", [0,1], CNOT_MAT),
    Gate("CX", [2,3], CNOT_MAT),
    Gate("CX", [1,2], CNOT_MAT),
])
bw.construct_layers()
for i, layer in enumerate(bw.layers):
    print(f"Layer {i}: {[str(g) for g in layer]}")

In [ ]:
# Composition and repetition — layers are cleared on copy()
ghz2  = ghz + ghz    # concatenate
ghz3  = ghz * 3      # repeat
ghz3r = 3 * ghz      # __rmul__

print(f"ghz + ghz : {len(ghz2)} gates, layers={ghz2.layers}")
print(f"ghz * 3   : {len(ghz3)} gates")
print(f"3 * ghz   : {len(ghz3r)} gates")

In [ ]:
# dagger: reverses gate order and daggers each gate
ghz_fresh = Circuit([Gate("H", 0, H_MAT), Gate("CX", [0,1], CNOT_MAT)])
ghz_dag   = ghz_fresh.dagger()

print("  Circuit gates:", [str(g) for g in ghz_fresh.gates])
print("Daggered gates:", [str(g) for g in ghz_dag.gates])
print("\nNote: order reversed, each gate has †")

In [ ]:
# Slicing returns a Circuit, indexing returns a Gate
circ = Circuit([Gate("H", i, H_MAT) for i in range(4)])
print("circ[1]   :", circ[1], type(circ[1]))
print("circ[1:3] :", circ[1:3], type(circ[1:3]))

---
## 3. Gate embedding — `_embed_gate`

Before a gate can be applied to a full-system state, its matrix must be lifted into the full Hilbert space. `_embed_gate` handles both the contiguous case (kron with identities) and the non-contiguous case (permutation → apply → permute back).

The non-contiguous case is the one worth verifying by eye.

In [ ]:
# Contiguous: H on qubit 0 of 2-qubit system → H ⊗ I
U = _embed_gate(Gate("H", 0, H_MAT), n=2, local_dim=2)
print("_embed_gate(H, 0) on 2 qubits:")
print(np.round(U.real, 4))
print("\nnp.kron(H, I):")
print(np.round(np.kron(H_MAT, np.eye(2)).real, 4))
print("\nMatch:", np.allclose(U, np.kron(H_MAT, np.eye(2))))

In [ ]:
# Non-contiguous: CNOT on qubits 0 and 2 of a 3-qubit system.
# Qubit 1 is a spectator — it just rides along.
# Control = qubit 0, target = qubit 2.
#
# Expected action on basis states (notation: |q0 q1 q2⟩):
#   |0 _ _⟩ → unchanged (control is 0)
#   |1 _ 0⟩ → |1 _ 1⟩  (control is 1, flip target)
#   |1 _ 1⟩ → |1 _ 0⟩

cx02 = Gate("CX", [0, 2], CNOT_MAT)
U = _embed_gate(cx02, n=3, local_dim=2)

test_states = {
    "000": 0b000,  # control=0 → unchanged
    "010": 0b010,  # control=0 → unchanged  
    "100": 0b100,  # control=1, target=0 → flip target → |101⟩
    "101": 0b101,  # control=1, target=1 → flip target → |100⟩
    "110": 0b110,  # control=1, target=0 → |111⟩
    "111": 0b111,  # control=1, target=1 → |110⟩
}

for label, idx in test_states.items():
    psi = np.zeros(8); psi[idx] = 1.0
    out_idx = np.argmax(np.abs(U @ psi))
    print(f"|{label}⟩  →  |{out_idx:03b}⟩")

In [ ]:
# Unitarity check: U†U = I
print("U†U = I?", np.allclose(U.conj().T @ U, np.eye(8), atol=1e-12))

# The full 8×8 matrix — mostly identity with the swap block at the bottom
print("\nCNOT([0,2]) on 3 qubits:")
print(np.real(U).astype(int))

---
## 4. Circuit unitary — `to_matrix`

`Circuit.to_matrix()` multiplies out all embedded gate matrices in order (first gate applied first). Useful for small circuits and sanity checks.

In [ ]:
ghz_circ = Circuit([Gate("H", 0, H_MAT), Gate("CX", [0, 1], CNOT_MAT)], n=2)
U_ghz = ghz_circ.to_matrix()

print("GHZ unitary:")
print(np.round(U_ghz.real, 4))

# |00⟩ should become (|00⟩ + |11⟩) / √2
psi_00  = np.array([1, 0, 0, 0], dtype=complex)
psi_out = U_ghz @ psi_00
print(f"\nU_ghz |00⟩ = {np.round(psi_out, 4)}")
print(f"Expected   = {np.round(np.array([1,0,0,1])/np.sqrt(2), 4)}")
print(f"Bell state? {np.allclose(psi_out, np.array([1,0,0,1])/np.sqrt(2))}")

In [ ]:
# GHZ†GHZ = I (circuit dagger undoes the circuit)
U_dag = ghz_circ.dagger().to_matrix()
print("U_ghz† U_ghz = I?", np.allclose(U_dag @ U_ghz, np.eye(4), atol=1e-12))

---
## 5. Observable

`Observable` is a pure descriptor: just `name` and `indices`. No matrix lives on the base class. `State.expectation()` dispatches on the concrete subclass and builds whatever representation it needs — the same pattern as `Simulator` dispatching on `Gate` type.

Concrete subclasses (`PauliString`, `Magnetization`, etc.) come in Task 6.

In [ ]:
# Observable is abstract (no instantiation) but trivial to subclass.
class MatrixObservable(Observable):
    """Fallback observable: stores a dense matrix."""
    def __init__(self, name, indices, matrix):
        super().__init__(name, indices)
        self.matrix = np.asarray(matrix, dtype=complex)

Z0 = MatrixObservable("Z", 0, Z_MAT)
ZZ = MatrixObservable("ZZ", [0, 1], np.kron(Z_MAT, Z_MAT))

print(Z0)
print(ZZ)
print("Z0.indices:", Z0.indices)
print("ZZ.matrix:\n", ZZ.matrix.real.astype(int))

---
## 6. NoiseModel

The base `NoiseModel` has three methods: `get_kraus` (raises `NotImplementedError` by default), `get_superop` (implemented in terms of `get_kraus` once the subclass provides it), and `sample_error` (raises `NotImplementedError`). None are `@abstractmethod` because not every noise model can provide every form.

In [ ]:
# Minimal depolarising noise model to show the interface
class DepolarNoise(NoiseModel):
    """Single-qubit depolarising channel with error probability p."""
    def __init__(self, p: float):
        self.p = p

    def get_kraus(self, gate):
        p = self.p
        return [
            np.sqrt(1 - p) * np.eye(2),
            np.sqrt(p/3) * X_MAT,
            np.sqrt(p/3) * np.array([[0,-1j],[1j,0]]),  # Y
            np.sqrt(p/3) * Z_MAT,
        ]

nm = DepolarNoise(p=0.1)
x_gate = Gate("X", 0, X_MAT)
kraus = nm.get_kraus(x_gate)

print(f"Kraus operators: {len(kraus)} matrices of shape {kraus[0].shape}")
print("CPTP check: sum(K†K) = I?")
print(np.round(sum(k.conj().T @ k for k in kraus).real, 6))

In [ ]:
# get_superop is inherited — works automatically once get_kraus is defined
superop = nm.get_superop(x_gate)
print(f"Superoperator shape: {superop.shape}")
print(np.round(superop.real, 4))

---
## 7. A minimal working backend

The real payoff of the abstractions is here. A 25-line statevector backend that uses everything above: subclasses `State` and `Simulator`, calls `_embed_gate` inside `translate_gate`, and runs an actual GHZ circuit end-to-end.

This is a sketch of what `backends/statevector.py` (Task 7) will look like.

In [ ]:
class NumpySV(State):
    """Dense statevector. Initialised from a computational-basis bitstring."""
    def __init__(self, n: int, bitstring: str = None):
        self.n = n
        self.data = np.zeros(2**n, dtype=complex)
        idx = int(bitstring, 2) if bitstring else 0
        self.data[idx] = 1.0

    def expectation(self, observable):
        # Works for MatrixObservable defined above; other types need a dispatcher.
        M = _embed_gate(Gate(observable.name, observable.indices, observable.matrix),
                        self.n, local_dim=2)
        return complex(self.data.conj() @ M @ self.data)

    def sample(self, shots: int):
        probs = np.abs(self.data)**2
        return np.random.choice(2**self.n, size=shots, p=probs)

    def sample_and_collapse(self, sites):
        raise NotImplementedError

    def overlap(self, other: "NumpySV") -> complex:
        return complex(self.data.conj() @ other.data)

    def __repr__(self):
        nonzero = [(f"|{i:0{self.n}b}⟩", a) for i, a in enumerate(self.data) if abs(a) > 1e-9]
        terms = " + ".join(f"{a:.4f}{ket}" for ket, a in nonzero)
        return terms


class NumpySVSim(Simulator):
    def _validate_state(self, state):
        if not isinstance(state, NumpySV):
            raise IncompatibleStateError(f"Expected NumpySV, got {type(state).__name__}")

    def translate_gate(self, gate):
        return _embed_gate(gate, self.circuit.n, self.circuit.local_dim)

    def _apply_translated_gate(self, state, matrix):
        state.data = matrix @ state.data

In [ ]:
# Run GHZ circuit
ghz_circ = Circuit([Gate("H", 0, H_MAT), Gate("CX", [0, 1], CNOT_MAT)], n=2)
sv       = NumpySV(2, bitstring="00")
result   = NumpySVSim(ghz_circ, sv).run()

print("Initial state:", sv)
print("After GHZ:    ", result)
print("Norm:", np.round(np.abs(result.data)**2).sum())

In [ ]:
# Key invariant: sv (the initial state) is unchanged after run();
# result is a deep copy that is fully independent.
print("Initial sv after run():", sv)        # still |00⟩
print("result is not sv:", result is not sv)
print("sv.data is not result.data:", sv.data is not result.data)

In [ ]:
# Feed-forward: apply X(0) to the Bell state → (|10⟩ + |01⟩)/√2
x_circ  = Circuit([Gate("X", 0, X_MAT)], n=2)
result2 = NumpySVSim(x_circ, result).run()
print("After X(0) on Bell state:", result2)

In [ ]:
# Expectation values: ⟨ZZ⟩ on the Bell state should be 1
ZZ_obs = MatrixObservable("ZZ", [0, 1], np.kron(Z_MAT, Z_MAT))
print("⟨ZZ⟩ on Bell state:", result.expectation(ZZ_obs).real)  # → 1.0
print("⟨ZZ⟩ on |00⟩:       ", sv.expectation(ZZ_obs).real)     # → 1.0
print("⟨ZZ⟩ on |10⟩:       ", NumpySV(2, bitstring="10").expectation(ZZ_obs).real)  # → -1.0

In [ ]:
# Sampling from the Bell state: should give ~50% |00⟩, ~50% |11⟩
samples = result.sample(shots=2000)
unique, counts = np.unique(samples, return_counts=True)
for val, cnt in zip(unique, counts):
    print(f"  |{val:02b}⟩: {cnt/2000:.3f}")

In [ ]:
# IncompatibleNoiseError: the default _insert_noise raises if nm is not None
try:
    NumpySVSim(ghz_circ, NumpySV(2), noise_model=DepolarNoise(p=0.01)).run()
except IncompatibleNoiseError as e:
    print(f"IncompatibleNoiseError: {e}")

In [ ]:
# IncompatibleStateError: passing wrong State type raises at construction, before run()
class SomeOtherState(State):
    def expectation(self, o): return 0.0
    def sample(self, s): return np.array([])
    def sample_and_collapse(self, s): return ("0", self)
    def overlap(self, o): return 0.0

try:
    NumpySVSim(ghz_circ, SomeOtherState())
except IncompatibleStateError as e:
    print(f"IncompatibleStateError: {e}")